# 03 · Selección de variables

**Qué hace este notebook:** puntúa cada predictor con varios criterios
independientes y los combina en un **ranking consensuado**. La salida es una
lista de variables seleccionadas, con la traza completa de por qué cada una entró
o salió.

**Por qué consensuado y no un único criterio:** cada método tiene un sesgo
conocido. Pearson no ve relaciones no lineales. La información mutua no ve
redundancia, así que premia dos veces a dos variables que dicen lo mismo. LASSO
elige arbitrariamente entre variables colineales -- y en datos meteorológicos casi
todas lo son. La importancia por impureza de un *random forest* favorece a las
variables de alta cardinalidad. Un ranking que sobrevive a los cinco es un
resultado; un ranking que sale de uno solo es un artefacto de ese sesgo.

**La regla, otra vez:** todo lo que sigue se calcula **sólo sobre `train`**.
Seleccionar variables mirando validación o test es fuga de información, y es de
las peores porque no deja rastro: el modelo final nunca ve esos datos, pero la
lista de variables con la que se entrena sí los vio.

| | |
|---|---|
| Lee | `data/gold/model_matrix.parquet`, `models/preprocessor.joblib` |
| Escribe | `reports/selection/consensus_ranking.csv`, `reports/selection/selected_features.json` |

**Cuándo pasa a `src/`:** cuando el conjunto de criterios y el umbral estén
fijados, esto se convierte en `src/packagename/features/selection.py`, con el
notebook reducido a llamarlo y mirar las figuras.

In [ ]:
DATASET = "model_matrix.parquet"

# Número de variables a retener. `None` usa el codo de la curva de importancia
# acumulada en lugar de un número decidido de antemano.
N_FEATURES: int | None = None

# Por encima de este VIF una variable se marca como redundante. 10 es la
# convención; 5 es lo que se usa cuando se quiere interpretar coeficientes.
VIF_LIMIT = 10.0

# Correlación por encima de la cual dos predictores se consideran el mismo, y
# sólo sobrevive el mejor situado en el consenso.
REDUNDANCY_LIMIT = 0.95

# Particiones de la validación cruzada temporal usada por los métodos que
# necesitan ajustar un modelo.
CV_SPLITS = 5

In [ ]:
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNetCV, LassoCV
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

from packagename import get_settings, set_seed, setup_logging
from packagename.etl import read_table, write_table
from packagename.viz import COLOR_NAMES, NEUTRALS, apply_style, remove_grid, savefig

setup_logging(level="INFO")
apply_style("paper")

settings = get_settings()
seed = set_seed(settings.random_seed)
settings.paths.ensure()

FIG = "selection"
TABLES = settings.paths.reports / "selection"
TABLES.mkdir(parents=True, exist_ok=True)

In [ ]:
artefact = joblib.load(settings.paths.models / "preprocessor.joblib")
TARGET = artefact["target"]
predictors = artefact["transformed_names"]

matrix = read_table(settings.paths.gold / DATASET).set_index("time").sort_index()

train = matrix[matrix["split"] == "train"]
X_train = train[predictors]
y_train = train[TARGET]

# La validación cruzada es temporal en todos los métodos de abajo. Un KFold
# aleatorio sobre una serie con memoria reparte instantes vecinos entre ajuste y
# evaluación, y entonces mide cuánto recuerda el modelo, no cuánto predice.
cv = TimeSeriesSplit(n_splits=CV_SPLITS)

print(f"{len(X_train):,} filas de entrenamiento, {len(predictors)} predictores")
print(f"Objetivo: {TARGET}")

## 1. Filtros univariantes

Baratos, y por eso van primero: descartan lo que no hace falta puntuar con nada
más caro. Miran una variable a la vez, así que no ven redundancia -- eso llega en
la sección 2.

In [ ]:
# Varianza casi nula. Tras el escalado de `02` la varianza de train es ~1 por
# construcción, así que lo que aparezca aquí son las columnas indicadoras de
# imputación con casi ningún hueco: constantes disfrazadas de variables.
variance = X_train.var()
near_zero = variance[variance < 0.01]

print(f"Varianza < 0.01: {len(near_zero)} columnas")
near_zero.sort_values().to_frame("varianza")

In [ ]:
univariate = pd.DataFrame(
    {
        "pearson": X_train.corrwith(y_train, method="pearson").abs(),
        "spearman": X_train.corrwith(y_train, method="spearman").abs(),
        "mi": mutual_info_regression(X_train, y_train, random_state=seed),
    },
    index=predictors,
)
# La diferencia entre Spearman y Pearson es la pista de no linealidad: si es
# grande, un baseline lineal va a dejar señal sin explotar en esa variable.
univariate["no_linealidad"] = univariate["spearman"] - univariate["pearson"]
univariate.sort_values("mi", ascending=False).round(3).head(20)

## 2. Métodos multivariantes

Estos sí ven el conjunto. El VIF cuantifica la redundancia sin ajustar nada
predictivo; LASSO y Elastic Net la resuelven eligiendo, y la diferencia entre
ambos es exactamente lo que hace la penalización L2: donde LASSO se queda con una
variable de un grupo colineal y anula las demás, Elastic Net reparte el
coeficiente entre todas. Comparar sus dos rankings es la forma más directa de ver
qué variables forman grupo.

In [ ]:
# -> src/packagename/features/diagnostics.py (compartida con 01)
def variance_inflation(frame: pd.DataFrame) -> pd.Series:
    correlation = frame.corr().to_numpy()
    return pd.Series(np.diag(np.linalg.pinv(correlation)), index=frame.columns)


vif = variance_inflation(X_train).sort_values(ascending=False)
print(f"VIF > {VIF_LIMIT}: {int((vif > VIF_LIMIT).sum())} de {len(vif)} predictores")
vif.head(15).round(2).to_frame("VIF")

In [ ]:
# `n_alphas` y `cv` explícitos: el camino de regularización se elige por
# validación cruzada temporal, no por el KFold aleatorio que es el defecto.
lasso = LassoCV(cv=cv, n_alphas=100, random_state=seed, n_jobs=-1, max_iter=5000)
lasso.fit(X_train, y_train)

elastic = ElasticNetCV(
    cv=cv,
    l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 1.0],
    random_state=seed,
    n_jobs=-1,
    max_iter=5000,
)
elastic.fit(X_train, y_train)

linear = pd.DataFrame(
    {"lasso": np.abs(lasso.coef_), "elastic_net": np.abs(elastic.coef_)},
    index=predictors,
)

print(f"LASSO:       alpha = {lasso.alpha_:.5f}, {int((lasso.coef_ != 0).sum())} coef. no nulos")
print(
    f"Elastic Net: alpha = {elastic.alpha_:.5f}, l1_ratio = {elastic.l1_ratio_}, "
    f"{int((elastic.coef_ != 0).sum())} coef. no nulos"
)
linear.sort_values("lasso", ascending=False).round(4).head(20)

In [ ]:
# El camino de coeficientes es más informativo que la lista final: se ve en qué
# orden entran las variables al relajar la penalización, y qué grupos entran
# juntos.
alphas = np.logspace(np.log10(lasso.alpha_ * 100), np.log10(lasso.alpha_ / 10), 60)
path = np.array(
    [
        LassoCV(alphas=[alpha], cv=cv, random_state=seed, max_iter=5000).fit(X_train, y_train).coef_
        for alpha in alphas
    ]
)

fig, ax = plt.subplots(figsize=(10, 5))
strongest = linear["lasso"].nlargest(10).index
for name in predictors:
    position = predictors.index(name)
    highlight = name in strongest
    ax.plot(
        alphas,
        path[:, position],
        lw=1.4 if highlight else 0.5,
        color=None if highlight else NEUTRALS["light"],
        label=name if highlight else None,
        zorder=3 if highlight else 1,
    )
ax.axvline(lasso.alpha_, color=NEUTRALS["dark_slate"], ls="--", lw=0.8)
ax.set_xscale("log")
ax.invert_xaxis()
ax.set_xlabel("alpha (penalización decreciente ->)")
ax.set_ylabel("coeficiente")
ax.set_title("Camino de regularización del LASSO")
ax.legend(ncols=2, fontsize=8)
savefig(fig, f"{FIG}/lasso_path.png")

## 3. Métodos basados en modelos

Un *random forest* y un *gradient boosting*, con dos medidas cada uno. La
distinción importa: la importancia por impureza se lee de la estructura del árbol
ya ajustado y está sesgada hacia las variables con más valores distintos; la
importancia por permutación mide la pérdida real de rendimiento al destruir una
variable, y por tanto responde a la pregunta que de verdad interesa.

La permutación se calcula sobre `valid` y no sobre `train`: sobre train mide
cuánto se apoyó el modelo en la variable *para memorizar*, que es otra pregunta.

In [ ]:
forest = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=seed,
    n_jobs=-1,
)
forest.fit(X_train, y_train)

booster = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=seed,
    n_jobs=-1,
)
booster.fit(X_train, y_train)

model_based = pd.DataFrame(
    {"random_forest": forest.feature_importances_, "xgboost": booster.feature_importances_},
    index=predictors,
)
model_based.sort_values("random_forest", ascending=False).round(4).head(20)

In [ ]:
valid = matrix[matrix["split"] == "valid"]

# n_repeats bajo a propósito: cada repetición es una pasada completa de
# predicción por variable, así que el coste crece con el producto. Subirlo cuando
# el ranking se estabilice y haga falta el error estándar.
permutation = permutation_importance(
    forest,
    valid[predictors],
    valid[TARGET],
    n_repeats=10,
    random_state=seed,
    n_jobs=-1,
    scoring="neg_root_mean_squared_error",
)

permuted = pd.DataFrame(
    {"permutation": permutation.importances_mean, "permutation_std": permutation.importances_std},
    index=predictors,
).sort_values("permutation", ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
top = permuted.head(25).iloc[::-1]
ax.barh(top.index, top["permutation"], xerr=top["permutation_std"], color=COLOR_NAMES["mint-green"])
ax.axvline(0, color=NEUTRALS["dark_slate"], lw=0.8)
ax.set_xlabel("aumento del RMSE al permutar")
ax.set_title("Importancia por permutación (medida en validación)")
savefig(fig, f"{FIG}/permutation_importance.png")

# Una importancia negativa significa que el modelo predice mejor sin esa
# variable: es ruido al que se ha ajustado.
print("Importancia <= 0:", permuted.index[permuted["permutation"] <= 0].tolist())

## 4. Ranking consensuado

Cada criterio se convierte en un rango (1 = más importante) y se combina con la
mediana en lugar de la media: así un único método discrepante no arrastra el
resultado, que es precisamente por lo que se usan varios.

La columna `desacuerdo` es la más informativa de la tabla. Un rango que varía
mucho entre criterios indica una variable colineal -- los métodos se reparten el
crédito de forma arbitraria dentro del grupo -- y por tanto una variable que hay
que decidir mirando el grupo entero y no su fila.

In [ ]:
scores = pd.concat(
    [
        univariate[["pearson", "spearman", "mi"]],
        linear,
        model_based,
        permuted[["permutation"]],
    ],
    axis=1,
)
# Todos los criterios son "más grande es mejor", así que el rango es descendente
# y homogéneo. Un criterio de signo contrario tendría que invertirse aquí, no en
# la tabla final.
ranks = scores.rank(ascending=False, method="average")
ranks["ranking_final"] = ranks.median(axis=1).rank(method="first").astype(int)
ranks["desacuerdo"] = scores.rank(ascending=False).std(axis=1)

consensus = (
    ranks.join(scores.add_prefix("valor_")).join(vif.to_frame("vif")).sort_values("ranking_final")
)
consensus.head(25).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
criteria = [
    "pearson",
    "spearman",
    "mi",
    "lasso",
    "elastic_net",
    "random_forest",
    "xgboost",
    "permutation",
]
shown = consensus.head(30)
mesh = ax.pcolormesh(shown[criteria].to_numpy(), cmap="viridis_r")
ax.set_xticks(np.arange(len(criteria)) + 0.5, criteria, rotation=45, ha="right")
ax.set_yticks(np.arange(len(shown)) + 0.5, shown.index)
ax.invert_yaxis()
ax.set_title("Rango por criterio (más claro = más importante)")
remove_grid(ax)
fig.colorbar(mesh, ax=ax, label="rango")
savefig(fig, f"{FIG}/consensus_heatmap.png")

## 5. Poda por redundancia

El consenso ordena, pero no elimina duplicados: dos variables que dicen lo mismo
aparecen las dos arriba. Esta celda recorre el ranking de mejor a peor y descarta
toda variable que esté correlacionada por encima de `REDUNDANCY_LIMIT` con
alguna ya aceptada. Al hacerlo en orden de consenso, el representante de cada
grupo es el mejor situado, no el primero por orden alfabético.

In [ ]:
# -> src/packagename/features/selection.py
def prune_redundant(frame: pd.DataFrame, order: list[str], limit: float) -> dict[str, str]:
    correlation = frame[order].corr().abs()
    decisions: dict[str, str] = {}
    kept: list[str] = []
    for name in order:
        collision = next((other for other in kept if correlation.loc[name, other] > limit), None)
        if collision is None:
            kept.append(name)
            decisions[name] = "retenida"
        else:
            decisions[name] = (
                f"redundante con {collision} (r={correlation.loc[name, collision]:.3f})"
            )
    return decisions


order = consensus.index.tolist()
decisions = prune_redundant(X_train, order, REDUNDANCY_LIMIT)
consensus["decision"] = pd.Series(decisions)

survivors = [name for name in order if decisions[name] == "retenida"]
print(f"{len(order)} -> {len(survivors)} predictores tras la poda por redundancia")
consensus.loc[consensus["decision"] != "retenida", ["ranking_final", "decision"]].head(20)

## 6. Cuántas quedarse

Con `N_FEATURES = None` el corte lo pone la curva de importancia acumulada por
permutación: se retienen las variables que suman el 95% de la importancia
positiva. Es preferible a un número redondo porque el número redondo es una
decisión disfrazada de dato.

In [ ]:
positive = permuted.loc[survivors, "permutation"].clip(lower=0)
cumulative = positive.sort_values(ascending=False).cumsum() / positive.sum()
elbow = int((cumulative < 0.95).sum()) + 1
n_keep = N_FEATURES or elbow

selected = [name for name in survivors if name in cumulative.index[:n_keep]]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.arange(1, len(cumulative) + 1), cumulative.to_numpy(), marker=".")
ax.axhline(0.95, color=NEUTRALS["light"], ls="--")
ax.axvline(n_keep, color=COLOR_NAMES["crimson"], ls="--", label=f"corte en {n_keep}")
ax.set_xlabel("número de predictores")
ax.set_ylabel("importancia acumulada")
ax.set_title("Dónde cortar")
ax.legend()
savefig(fig, f"{FIG}/cumulative_importance.png")

print(f"Codo del 95%: {elbow} predictores. Seleccionados: {len(selected)}")
selected

## 7. ¿Merece la pena la poda?

Una selección que empeora el modelo no es una selección, es una pérdida. Esta
celda compara en validación el mismo *random forest* con todos los predictores y
con los seleccionados. Se espera un RMSE parecido o mejor: si empeora
claramente, el corte de la sección 6 es demasiado agresivo.

In [ ]:
def rmse_on_valid(columns: list[str]) -> float:
    model = RandomForestRegressor(
        n_estimators=300, min_samples_leaf=5, random_state=seed, n_jobs=-1
    )
    model.fit(X_train[columns], y_train)
    error = valid[TARGET].to_numpy() - model.predict(valid[columns])
    return float(np.sqrt(np.mean(error**2)))


full_rmse = rmse_on_valid(predictors)
selected_rmse = rmse_on_valid(selected)

print(f"RMSE con {len(predictors)} predictores: {full_rmse:.4f}")
print(f"RMSE con {len(selected)} predictores:  {selected_rmse:.4f}")
print(f"Cambio: {100 * (selected_rmse - full_rmse) / full_rmse:+.2f}%")

In [ ]:
write_table(consensus.reset_index(names="variable"), TABLES / "consensus_ranking.csv")

(TABLES / "selected_features.json").write_text(
    json.dumps(
        {
            "target": TARGET,
            "selected": selected,
            "n_candidates": len(predictors),
            "criteria": criteria,
            "params": {
                "N_FEATURES": N_FEATURES,
                "VIF_LIMIT": VIF_LIMIT,
                "REDUNDANCY_LIMIT": REDUNDANCY_LIMIT,
                "CV_SPLITS": CV_SPLITS,
            },
            "valid_rmse": {"all_features": full_rmse, "selected": selected_rmse},
        },
        indent=2,
    )
)
print(TABLES / "selected_features.json")

## Conclusiones

A rellenar antes de pasar a `04`:

1. **Qué variables entran, y por qué** -- citando el `ranking_final` y la columna
   `decision` del CSV.
2. **Qué grupos colineales se han resuelto** y qué representante se ha quedado:
   son las filas con `desacuerdo` alto.
3. **Qué variables tienen importancia por permutación negativa**, porque son ruido
   al que el modelo se ajustó y deberían haber caído.
4. **Si la poda ha costado rendimiento** (§7) y si el precio es aceptable a cambio
   de un modelo interpretable.

Dos cosas que este notebook **no** decide, para no confundirlas más adelante:

- **La importancia final del modelo elegido.** Aquí se usan RF, XGBoost y
  permutación como *criterios de selección*, sobre modelos rápidos y sin ajustar
  hiperparámetros. Explicar el modelo definitivo es `05`.
- **Qué modelo es mejor.** Los ajustes de este notebook existen para puntuar
  variables. Comparar modelos es `04`, sobre las variables que salen de aquí.